In [0]:
spark.sql("DROP SCHEMA IF EXISTS main_workspace.silver CASCADE")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS main_workspace.silver
""")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS main_workspace.silver_quarantine
""")

In [0]:
df_customers_raw = spark.table("main_workspace.bronze.customers")

display(df_customers_raw)

print("Total Records:", df_customers_raw.count())

df_customers_raw.printSchema()

# STEP 2 — Identify Column Types

In [0]:
string_cols = [f.name for f in df_customers_raw.schema.fields if f.dataType.simpleString() == 'string']
numeric_cols = [f.name for f in df_customers_raw.schema.fields if f.dataType.simpleString() in ['int','bigint','double','float']]

In [0]:
print("String Columns:", string_cols)
print("Numeric Columns:", numeric_cols)

## # STEP 3 — Detect Whitespace Issues (ALL STRING COLUMNS)

In [0]:
for col_name in string_cols:
    print(f"Checking whitespace issues in {col_name}")
    
    display(
        df_customers_raw.filter(
            (col(col_name).startswith(" ")) |
            (col(col_name).endswith(" "))
        )
    )

In [0]:
df_trim = df_customers_raw

for col_name in string_cols:
    df_trim = df_trim.withColumn(col_name, trim(col(col_name)))

In [0]:
display(df_trim)

In [0]:
for col_name in string_cols:
    display(
        df_trim.filter(
            (col(col_name).startswith(" ")) |
            (col(col_name).endswith(" "))
        )
    )

# STEP 5 — Detect Empty Strings

In [0]:
for col_name in string_cols:
    print(f"Empty string count in {col_name}")
    
    display(
        df_trim.filter(col(col_name) == "")
    )

In [0]:
df_empty_to_null = df_trim

for col_name in string_cols:
    df_empty_to_null = df_empty_to_null.withColumn(
        col_name,
        when(col(col_name) == "", None).otherwise(col(col_name))
    )

In [0]:
display(df_empty_to_null)

# STEP 7 — Detect Special Characters

Example invalid patterns:

$
#
?
unknown

In [0]:
for col_name in string_cols:
    print(f"Checking invalid characters in {col_name}")
    
    display(
        df_empty_to_null.filter(
            col(col_name).rlike("[^a-zA-Z0-9 ]")
        )
    )

In [0]:
df_invalid_clean = df_empty_to_null

for col_name in string_cols:
    df_invalid_clean = df_invalid_clean.withColumn(
        col_name,
        when(col(col_name).rlike("[^a-zA-Z0-9 ]"), None).otherwise(col(col_name))
    )

In [0]:
display(df_invalid_clean)

In [0]:
display(
df_invalid_clean.select([
count(when(col(c).isNull(), c)).alias(c)
for c in df_invalid_clean.columns
])
)

# STEP 10 — Replace NULL Values


In [0]:
df_null_fixed = df_invalid_clean

fill_dict = {}

for c in string_cols:
    fill_dict[c] = "None"

for c in numeric_cols:
    fill_dict[c] = 0

df_null_fixed = df_null_fixed.fillna(fill_dict)

In [0]:
display(
df_null_fixed.select([
count(when(col(c).isNull(), c)).alias(c)
for c in df_null_fixed.columns
])
)

In [0]:
display(df_null_fixed)

## STEP 11 — Detect Duplicates

In [0]:
display(
df_null_fixed.groupBy("customer_id")
.count()
.filter(col("count") > 1)
)

## STEP 12 — Remove Duplicates

In [0]:
window_spec = Window.partitionBy("customer_id").orderBy(desc("customer_zip_code_prefix"))

df_dedup = df_null_fixed \
.withColumn("row_num", row_number().over(window_spec)) \
.filter(col("row_num") == 1) \
.drop("row_num")

In [0]:
display(
df_dedup.groupBy("customer_id")
.count()
.filter(col("count") > 1)
)

## STEP 13 — Add Metadata Columns

In [0]:
display(df_dedup)

In [0]:
df_customers_final = df_dedup \
.withColumn("processed_timestamp", current_timestamp()) \
.withColumn("data_quality_flag", lit("VALID"))

In [0]:
display(df_customers_final)

In [0]:
display(
df_dedup.select("random_val").distinct()
)

In [0]:
df_clean_base = df_dedup.drop("random_val")

display(df_clean_base)
df_clean_base.printSchema()

Databricks filtered table. Run in Databricks to view.

In [0]:
string_cols = [
    f.name for f in df_clean_base.schema.fields
    if f.dataType.simpleString() == "string"
]

numeric_cols = [
    f.name for f in df_clean_base.schema.fields
    if f.dataType.simpleString() in ["int","bigint","double","float"]
]

In [0]:
print("String Columns:", string_cols)
print("Numeric Columns:", numeric_cols)

In [0]:
from functools import reduce

conditions = []

# NULL values
for c in df_clean_base.columns:
    conditions.append(col(c).isNull())

# string "None"
for c in string_cols:
    conditions.append(col(c) == "None")

# numeric zero
for c in numeric_cols:
    conditions.append(col(c) == 0)

quarantine_condition = reduce(lambda x, y: x | y, conditions)

In [0]:
customer_quarantine = df_clean_base.filter(quarantine_condition)

display(customer_quarantine)

print("Quarantine rows:", customer_quarantine.count())

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS main_workspace.quarantine_data
""")

In [0]:
customer_quarantine.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
"main_workspace.quarantine_data.customer_quarantine_data"
)

In [0]:
valid_customers = df_clean_base.subtract(customer_quarantine)

display(valid_customers)

print("Valid rows:", valid_customers.count())

In [0]:
valid_customers = valid_customers \
.withColumn("processed_timestamp", current_timestamp()) \
.withColumn("data_quality_flag", lit("VALID"))

In [0]:
valid_customers_cast = valid_customers \
.withColumn("customer_id", col("customer_id").cast("string")) \
.withColumn("customer_unique_id", col("customer_unique_id").cast("string")) \
.withColumn("customer_zip_code_prefix", col("customer_zip_code_prefix").cast("int")) \
.withColumn("customer_city", col("customer_city").cast("string")) \
.withColumn("customer_state", col("customer_state").cast("string"))

In [0]:
valid_customers_cast = valid_customers_cast \
.withColumn("processed_timestamp", current_timestamp()) \
.withColumn("data_quality_flag", lit("VALID"))

In [0]:
valid_customers_cast.printSchema()

display(valid_customers_cast)

In [0]:
valid_customers_cast.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("main_workspace.silver.customers")

In [0]:
customer_quarantine_cast = customer_quarantine \
.withColumn("customer_id", col("customer_id").cast("string")) \
.withColumn("customer_unique_id", col("customer_unique_id").cast("string")) \
.withColumn("customer_zip_code_prefix", col("customer_zip_code_prefix").cast("int")) \
.withColumn("customer_city", col("customer_city").cast("string")) \
.withColumn("customer_state", col("customer_state").cast("string"))

In [0]:
customer_quarantine_cast = customer_quarantine_cast \
.withColumn("processed_timestamp", current_timestamp()) \
.withColumn("data_quality_flag", lit("QUARANTINE"))

In [0]:
customer_quarantine_cast.printSchema()

display(customer_quarantine_cast)

In [0]:
customer_quarantine_cast.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("main_workspace.quarantine_data.customer_quarantine_data")

# Orders Table – Silver Processing

In [0]:
df_orders_raw = spark.table("main_workspace.bronze.orders")

display(df_orders_raw)

print("Total Records:", df_orders_raw.count())

df_orders_raw.printSchema()

In [0]:
string_cols = [
    f.name for f in df_orders_raw.schema.fields
    if f.dataType.simpleString() == "string"
]

print("String columns:", string_cols)

## STEP 3 — Detect Whitespace Issues

In [0]:
for c in string_cols:
    print(f"Checking whitespace in {c}")
    display(
        df_orders_raw.filter(
            (col(c).startswith(" ")) |
            (col(c).endswith(" "))
        )
    )

## STEP 4 — Trim Whitespace

In [0]:
df_orders_trim = df_orders_raw

for c in string_cols:
    df_orders_trim = df_orders_trim.withColumn(c, trim(col(c)))

In [0]:
display(df_orders_trim)

## STEP 5 — Convert Empty Strings → NULL

In [0]:
df_orders_clean = df_orders_trim

for c in string_cols:
    df_orders_clean = df_orders_clean.withColumn(
        c,
        when(col(c) == "", None).otherwise(col(c))
    )

In [0]:
display(df_orders_clean)

## STEP 6 — Detect NULL Values

In [0]:
display(
df_orders_clean.select([
count(when(col(c).isNull(), c)).alias(c)
for c in df_orders_clean.columns
])
)

## STEP 7 — Build Quarantine Condition

Rows containing NULL values should go to quarantine.

In [0]:
from functools import reduce

conditions = []

for c in df_orders_clean.columns:
    conditions.append(col(c).isNull())

quarantine_condition = reduce(lambda x, y: x | y, conditions)

In [0]:
orders_quarantine = df_orders_clean.filter(quarantine_condition)

display(orders_quarantine)

print("Quarantine rows:", orders_quarantine.count())

In [0]:
valid_orders = df_orders_clean.subtract(orders_quarantine)

display(valid_orders)

print("Valid rows:", valid_orders.count())

In [0]:
orders_quarantine_cast = orders_quarantine \
.withColumn("order_id", col("order_id").cast("string")) \
.withColumn("customer_id", col("customer_id").cast("string")) \
.withColumn("order_status", col("order_status").cast("string")) \
.withColumn("order_purchase_timestamp", col("order_purchase_timestamp").cast("timestamp")) \
.withColumn("order_approved_at", col("order_approved_at").cast("timestamp")) \
.withColumn("order_delivered_carrier_date", col("order_delivered_carrier_date").cast("timestamp")) \
.withColumn("order_delivered_customer_date", col("order_delivered_customer_date").cast("timestamp")) \
.withColumn("order_estimated_delivery_date", col("order_estimated_delivery_date").cast("timestamp")) \
.withColumn("processed_timestamp", current_timestamp()) \
.withColumn("data_quality_flag", lit("QUARANTINE"))

In [0]:
orders_quarantine_cast = orders_quarantine_cast.withColumn('timestamp_column', functions.expr("try_cast(timestamp_column as timestamp)") )orders_quarantine_cast.printSchema()

display(orders_quarantine_cast)



In [0]:
valid_orders_cast = valid_orders \
.withColumn("order_id", col("order_id").cast("string")) \
.withColumn("customer_id", col("customer_id").cast("string")) \
.withColumn("order_status", col("order_status").cast("string")) \
.withColumn("order_purchase_timestamp", expr("try_cast(order_purchase_timestamp as timestamp)")) \
.withColumn("order_approved_at", expr("try_cast(order_approved_at as timestamp)")) \
.withColumn("order_delivered_carrier_date", expr("try_cast(order_delivered_carrier_date as timestamp)")) \
.withColumn("order_delivered_customer_date", expr("try_cast(order_delivered_customer_date as timestamp)")) \
.withColumn("order_estimated_delivery_date", expr("try_cast(order_estimated_delivery_date as timestamp)")) \
.withColumn("processed_timestamp", current_timestamp()) \
.withColumn("data_quality_flag", lit("VALID"))

In [0]:
display(
valid_orders_cast.select([
count(when(col(c).isNull(), c)).alias(c)
for c in valid_orders_cast.columns
])
)

In [0]:
orders_quarantine_cast.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("main_workspace.quarantine_data.orders_quarantine_data")

In [0]:
valid_orders_cast.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("main_workspace.silver.orders")

In [0]:
from pyspark.sql.functions import col, to_timestamp, current_timestamp, lit

orders_quarantine_cast = orders_quarantine \
.withColumn("order_id", col("order_id").cast("string")) \
.withColumn("customer_id", col("customer_id").cast("string")) \
.withColumn("order_status", col("order_status").cast("string")) \
.withColumn("order_purchase_timestamp", to_timestamp(col("order_purchase_timestamp"))) \
.withColumn("order_approved_at", to_timestamp(col("order_approved_at"))) \
.withColumn("order_delivered_carrier_date", to_timestamp(col("order_delivered_carrier_date"))) \
.withColumn("order_delivered_customer_date", to_timestamp(col("order_delivered_customer_date"))) \
.withColumn("order_estimated_delivery_date", to_timestamp(col("order_estimated_delivery_date"))) \
.withColumn("processed_timestamp", current_timestamp()) \
.withColumn("data_quality_flag", lit("QUARANTINE"))

In [0]:
orders_quarantine_cast.printSchema()

display(orders_quarantine_cast)

In [0]:
from pyspark.sql.functions import col, when, to_timestamp, current_timestamp, lit

timestamp_cols = [
"order_purchase_timestamp",
"order_approved_at",
"order_delivered_carrier_date",
"order_delivered_customer_date",
"order_estimated_delivery_date"
]

orders_quarantine_clean = orders_quarantine

for c in timestamp_cols:
    orders_quarantine_clean = orders_quarantine_clean.withColumn(
        c,
        when(col(c) == "not_available", None).otherwise(col(c))
    )

In [0]:
when(col(c).isin("not_available","unknown","None",""), None)

In [0]:
orders_quarantine_cast = orders_quarantine_clean \
.withColumn("order_id", col("order_id").cast("string")) \
.withColumn("customer_id", col("customer_id").cast("string")) \
.withColumn("order_status", col("order_status").cast("string")) \
.withColumn("order_purchase_timestamp", to_timestamp(col("order_purchase_timestamp"))) \
.withColumn("order_approved_at", to_timestamp(col("order_approved_at"))) \
.withColumn("order_delivered_carrier_date", to_timestamp(col("order_delivered_carrier_date"))) \
.withColumn("order_delivered_customer_date", to_timestamp(col("order_delivered_customer_date"))) \
.withColumn("order_estimated_delivery_date", to_timestamp(col("order_estimated_delivery_date"))) \
.withColumn("processed_timestamp", current_timestamp()) \
.withColumn("data_quality_flag", lit("QUARANTINE"))

In [0]:
orders_quarantine_cast.printSchema()

In [0]:
orders_quarantine_cast.show(10, False)

In [0]:
orders_quarantine_cast.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("main_workspace.quarantine_data.orders_quarantine_data")